# tunekit — fine-tune an open LLM in Colab

One-command LoRA/QLoRA fine-tuning. This notebook:
1. installs tunekit
2. lets you upload a dataset (or uses the bundled example)
3. trains a LoRA adapter on the free T4 GPU
4. chats with the result and (optionally) exports it

**Runtime → Change runtime type → T4 GPU** before you start.

In [ ]:
!pip install -q "tunekit[quant] @ git+https://github.com/Dolmaa24/tunekit"
!tunekit info

## 1. Data

Any of these formats work, tunekit auto-detects them:

| format | shape |
|---|---|
| messages | `{"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}` |
| alpaca | `{"instruction": "...", "input": "...", "output": "..."}` |
| sharegpt | `{"conversations": [{"from": "human", "value": "..."}, {"from": "gpt", "value": "..."}]}` |
| prompt/completion | `{"prompt": "...", "completion": "..."}` |
| text | `{"text": "..."}` |

Upload a `.jsonl` / `.csv` / `.parquet` file, or use a Hub dataset id (e.g. `yahma/alpaca-cleaned`).

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick your file; skip this cell to use the example below
DATA = next(iter(uploaded)) if uploaded else None
if DATA is None:
    !curl -sL -o pirate.jsonl https://raw.githubusercontent.com/Dolmaa24/tunekit/main/examples/pirate.jsonl
    DATA = "pirate.jsonl"
print("using", DATA)

In [ ]:
!tunekit validate --model Qwen/Qwen2.5-0.5B-Instruct --data {DATA}

## 2. Train

Pick a model that fits the GPU. On a free T4 (15 GB):

| model | setting | VRAM |
|---|---|---|
| `Qwen/Qwen2.5-0.5B-Instruct` | LoRA bf16 | ~4 GB |
| `Qwen/Qwen2.5-1.5B-Instruct` | LoRA bf16 | ~8 GB |
| `meta-llama/Llama-3.2-3B-Instruct` | `--set model.load_in_4bit=true` | ~6 GB |
| `Qwen/Qwen2.5-7B-Instruct` | `--set model.load_in_4bit=true` | ~12 GB |

In [ ]:
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
!tunekit train --model {MODEL} --data {DATA} --output outputs/run \
    --set train.epochs=2 --set train.batch_size=4 --set train.grad_accum=4 --set data.max_length=1024

## 3. Try it

In [ ]:
from tunekit.inference import load_for_inference, stream_reply
model, tok = load_for_inference("outputs/run")
def ask(q):
    print("".join(stream_reply(model, tok, [{"role": "user", "content": q}], max_new_tokens=200)))
ask("What is the capital of France?")

## 4. Export (optional)

Merge the adapter into the base model, then convert to GGUF for Ollama / LM Studio / llama.cpp.

In [ ]:
!tunekit merge outputs/run outputs/merged
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp && pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!tunekit export outputs/merged --quant q8_0 --llama-cpp llama.cpp --ollama
!ls -lh outputs/merged/*.gguf

In [ ]:
# Download the adapter (small) to your machine
!zip -qr adapter.zip outputs/run -x "outputs/run/checkpoint-*"
from google.colab import files; files.download("adapter.zip")